In [ ]:
import random
import math

from functools import reduce
from itertools import combinations

In [2]:
eps = 1.0e-14
N = 16
seed = 42

In [ ]:
random.seed(seed)

In [ ]:
class DifferentialEvolution:
    def __init__(self,
                 objective_function,
                 constraint_functions,
                 upper_bounds,
                 lower_bounds,
                 population_size):
        self.objective_function = objective_function
        self.constraint_functions = constraint_functions
        self.upper_bounds = upper_bounds
        self.lower_bounds = lower_bounds
        self.population_size = population_size

    def run(self,
            stopping_condition,
            crossover,
            selection_mutation):
        self.initialize_stats()

        population = [[self.new_chromosome(upper, lower) for upper, lower in zip(self.upper_bounds, self.lower_bounds)] for _ in range(self.population_size)]
        scores = [self.evaluate(x) for x in population]
        self.update_stats(population, scores)

        while stopping_condition(self):
            new_generation = []
            new_scores = []

            for x, s in zip(population, scores):
                u = crossover(x)
                v = selection_mutation(x,
                                       population,
                                       self.best_solution_hist[-1],
                                       u)
                v = self.enforce_bounds(v)
                f = self.evaluate(v)

                if f < s:
                    new_generation.append(v)
                    new_scores.append(s)
                else:
                    new_generation.append(x)
                    new_scores.append(s)
            population = new_generation
            scores = new_scores
            self.update_stats(population, scores)
        
        return self

    def initialize_stats(self):
        self.iter_count = 0
        self.eval_count = 0
        self.best_solution_hist = []
        self.best_score_hist = []
        self.population_diversity_hist = []

    def update_stats(self, population, scores):
        best_idx = scores.index(min(scores))

        self.best_solution_hist.append(population[best_idx])
        self.best_score_hist.append(scores[best_idx])
        self.population_diversity_hist.append(self.diversity(population))

    def new_chromosome(self, upper, lower):
        return lower + random.random()*(upper - lower)
    
    def enforce_bounds(self, x):
        return [a if lower<=a and a<=upper else self.new_chromosome(upper, lower) for a, upper, lower in zip(x, self.upper_bounds, self.lower_bounds)]
    
    def evaluate(self, x):
        self.eval_count += 1

        cs = [cf(x) for cf in self.constraint_functions]
        cs = [c if eps < c else 1 for c in cs]
        return self.objective_function(x) * reduce(lambda x, y: x * y, cs)

    def diversity(self, population):
        n = self.population_size
        total_dist = 0
        for a, b in combinations(population, r=2):
            total_dist += math.sqrt(sum([(x - y)**2 for x, y in zip(a, b)]))
        return total_dist/(n*(n-1)/2)